# Building a Deep Q-Network with Keras

### Set up the environment

In [1]:
!pip install gym

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import numpy as np
import gym

# create the evironment
env = gym.make('CartPole-v1')

# Set random seed for reproducibility
np.random.seed(42)
env.reset(seed=42)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


(array([ 0.0273956 , -0.00611216,  0.03585979,  0.0197368 ], dtype=float32),
 {})

### Define the DQN model

In [3]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam

In [4]:
def build_model(state_size,action_size):
    model = Sequential()
    model.add(Dense(24, input_dim= state_size,activation='relu'))
    model.add(Dense(24,activation='relu'))
    model.add(Dense(action_size,activation='linear'))
    model.compile(loss='mse',optimizer=Adam(learning_rate=0.001))
    return model

state_size = env.observation_space.shape[0]
action_size = env.action_space.n
model = build_model(state_size,action_size)

C:\Users\ilyass\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


### Implement the replay buffer

In [5]:
from collections import deque
import random

memory = deque(maxlen=2000)

def remember(state,action,reward,next_state,done):
    memory.append((state,action,reward,next_state,done))

### Implement the epsilon-greedy policy

In [6]:
epsilon = 1.0
epsilon_min = 0.01
epsilon_decay = 0.995

def act(state):
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)
    q_values = model.predict(state)
    return np.argmax(q_values[0])

### Implement the Q-Learning update

In [7]:
def replay(batch_size):
    global epsilon
    minibatch = random.sample(memory,batch_size)
    for state,action,reward,next_state,done in minibatch:
        target = reward
        if not done:
            target = reward + gamma * np.amax(model.predict(next_state)[0])
        target_f = model.predict(state)
        target_f[0][action] = target
        model.fit(state,target_f,epochs=1,verbose=0)
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay

### Train the DQN

In [10]:
# Training loop 
epsides = 50 # More epsilon to ensure sufficient training
batch_size = 32 # Mini-batch size for replay training 
gamma = 0.95 # Discount factor for future reward 

for e in range(epsides):
    state = env.reset()
    if isinstance(state,tuple): # Handle tuple oytput
        state = state[0]
    state = np.reshape(state,[1,state_size])

    for time in range(200): # Max step per epislon
        # Choose action using epsilon-greedy policy
        action = act(state)

        # Perform action in the environment
        result = env.step(action)        
        if len(result) == 4: # Handle 4-value output
            next_state , reward , done , _ = result
        else: # Handle 5-value output
            next_state, reward, done, _ , _ = result

        if isinstance(next_state, tuple): # Handle tuple next_state
            next_state = next_state[0]
        next_state = np.reshape(next_state,[1,state_size])

        # Store exprience in memory
        remember(state,action,reward,next_state,done)

        # Update state
        state = next_state

        if done: # If epsilon ends
            print(f"epsides: {e+1}/{epsides}, score: {time}, epsilon: {epsilon:.2}")
            break

        # Train the model using replay memory
    if len(memory) > batch_size:
        replay(batch_size)

env.close()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
epsides: 1/50, score: 10, epsilon: 0.92
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━

### Evaluate the performance

In [11]:
evaluation_episodes = 10
scores = []

for e in range(evaluation_episodes):
    state = env.reset()
    if isinstance(state, tuple):
        state = state[0]
    state = np.reshape(state,[1,state_size])

    total_reward = 0

    for time in range(200):
        action = np.argmax(model.predict(state)[0])

        result = env.step(action)
        if len(result) == 4:
            next_state,reward,done,_ = result
        else:
            next_state,reward,done,_,_ = result
        if isinstance(next_state,tuple):
            next_state = next_state[0]
        next_state = np.reshape(next_state,[1,state_size])

        state = next_state
        total_reward += reward

        if done:
            print(f"Evaluation episode: {e+1}/{evaluation_episodes},Score: {time}, Total reward: {total_reward}")
            scores.append(total_reward)
            break

print(f"Average reward: {np.mean(scores):.2f}, Max reward: {np.max(scores)}, Min reward: {np.min(scores)}")
env.close()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
Evaluation episode: 1/10,Score: 9, Total reward: 10.0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step
Evaluation episode: 2/10,Score: 9, Total reward: 10.0
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
1/1 ━━━━━━━━━━━━━━